# 1. Les visualisations, avant la prédiction

Ce notebook sert à regarder les données avant de lancer les modèles : qui est retiré, où sont les valeurs aberrantes, quelles colonnes sont liées au salaire.

Il travaille sur le même tableau que `2_Prediction` et `3_Prediction_Par_Cohorte` : les trois lisent leurs réglages dans `reglages.py`.

Pour changer un réglage (les anomalies à retirer, les valeurs aberrantes, les colonnes du modèle) : ouvrir `reglages.py`, le modifier, l'enregistrer, puis relancer ce notebook depuis sa première cellule.

---
## 1. Le tableau

La détection d'anomalies tourne d'abord : chaque salarié reçoit sa grille, son marché, un score et une priorité. Puis on calcule les ratios, et on retire les salariés choisis dans `reglages.py`.

| Colonne | Calcul | Lecture |
|---|---|---|
| `Compa_Ratio` | salaire / `Mid` | une des deux variables à prédire ; l'autre est le salaire fixe, `Fixe_Annuel_MAD` |
| `Ratio_Grille_Marche` | `Mid` / `Median` | 1,10 = la grille paie 10 % de plus que le marché |
| `Age_Embauche` | âge − ancienneté | l'expérience acquise avant d'arriver |

Les lignes affichées sous la cellule disent, dans l'ordre : combien de salariés sont lus, combien d'anomalies sont retirées (réglage `A_RETIRER`), combien de valeurs aberrantes sont écartées (réglage `ENLEVER`), et combien de salariés restent.

In [ ]:
import importlib

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

# Les trois fichiers communs du dossier : les réglages, la préparation du tableau, les modèles.
# importlib.reload les relit : un réglage changé dans reglages.py est pris en compte dès qu'on relance cette cellule.
import reglages, preparation, modeles
for fichier in (reglages, preparation, modeles):
    importlib.reload(fichier)
from preparation import CIBLES, RESULTATS, choisir_colonnes, outliers, preparer_tableau
from modeles import creer_modeles, metriques, salaire_predit

px.defaults.template = 'plotly_white'   # des graphes sur fond blanc
pd.set_option('display.precision', 2)   # les tableaux affichent 2 chiffres après la virgule ; les calculs gardent toute leur précision

tableau = preparer_tableau()
df = tableau.df   # les salariés gardés : ceux sur lesquels les modèles travailleront

print(f"\nColonnes du tableau : {list(df.columns)}")
df.head()

---
## 2. Les anomalies

Le premier tableau compte les salariés de chaque priorité, de la plus grave à la moins grave. Les priorités de la liste `A_RETIRER` sont celles qu'on retire.

Le deuxième montre les salariés retirés, du score le plus haut au plus bas.

Deux fichiers Excel sont écrits dans le dossier `resultats` : `anomalies.xlsx` (le résultat complet de la détection) et `salaries_retires.xlsx` (les salariés retirés, avec tout ce que la détection dit d'eux).

In [ ]:
# Combien de salariés dans chaque priorité, et lesquelles sont retirées
priorites = tableau.detection['Severity'].value_counts().reindex(tableau.priorites).fillna(0).astype(int).to_frame('Salariés')
priorites['Retirée'] = ['oui' if p in reglages.A_RETIRER else 'non' for p in priorites.index]
display(priorites)

# Les salariés retirés
retires = tableau.retires
print(f"{len(retires)} salariés retirés sur {len(tableau.detection)} ({len(retires) / len(tableau.detection):.1%}).")
if retires.empty:
    print("Personne n'est retiré : la suite se fait sur tous les salariés.")
a_montrer = ['Matricule', 'Job_Family', 'Niveau', 'Fixe_Annuel_MAD', 'Min', 'Mid', 'RiskScore', 'Severity', 'Reason_Principale']
display(retires[[c for c in a_montrer if c in retires.columns]].sort_values('RiskScore', ascending=False).head(10))

# Les deux fichiers Excel, dans le dossier resultats (2 chiffres après la virgule dans les fichiers)
tableau.detection.round(2).to_excel(RESULTATS / 'anomalies.xlsx', index=False)
retires.round(2).to_excel(RESULTATS / 'salaries_retires.xlsx', index=False)
print("Enregistré dans resultats : anomalies.xlsx et salaries_retires.xlsx")

---
## 3. Les valeurs aberrantes

Pour chaque colonne de `COLONNES_RETRAIT`, une boîte à moustaches. Les points au-delà des moustaches sont les outliers.

Les boîtes sont tracées sur les salariés qui restent une fois les anomalies retirées, avant tout retrait de valeur aberrante : on voit donc les outliers, que `ENLEVER` soit sur `True` ou sur `False`.

In [ ]:
avant = tableau.avant_outliers   # les salariés une fois les anomalies retirées, avant le retrait des valeurs aberrantes
colonnes_num = [c for c in reglages.COLONNES_RETRAIT if c in avant.columns]

bilan = pd.DataFrame({'Outliers': [int(outliers(avant[c]).sum()) for c in colonnes_num],
                      'Cases vides': [int(avant[c].isna().sum()) for c in colonnes_num]}, index=colonnes_num)
bilan['% outliers'] = (bilan['Outliers'] / len(avant) * 100).round(1)
display(bilan)

fig = px.box(avant[colonnes_num].melt(var_name='Colonne', value_name='Valeur'), x='Valeur', facet_col='Colonne',
             facet_col_wrap=3, height=220 * -(-len(colonnes_num) // 3), title='Une boîte à moustaches par colonne')
fig.update_xaxes(matches=None, showticklabels=True, title_text='')
# Au-dessus de chaque boîte, le nom de la colonne seul (et non « Colonne=Age »)
fig.for_each_annotation(lambda titre: titre.update(text=titre.text.split('=')[-1]))
fig.show()

Traitement, avec `ENLEVER = True` dans `reglages.py` : on écarte la ligne. Un salarié qui a un outlier dans une des colonnes de `COLONNES_RETRAIT` sort du tableau, même si ses autres colonnes sont normales.

Avec `ENLEVER = False`, les outliers restent dans le tableau : personne n'est écarté à cette étape.

Ce réglage n'a rien à voir avec les anomalies de l'étape 2 : une anomalie vient de la détection, un outlier vient seulement de la boîte à moustaches.

In [ ]:
ecartes = tableau.ecartes
print(f"ENLEVER = {reglages.ENLEVER} : {len(avant)} salariés au départ, {len(ecartes)} écartés ({len(ecartes) / len(avant):.1%}), {len(df)} gardés.")

if not ecartes.empty:
    # Combien de lignes pour chaque raison (une ligne qui a deux raisons est comptée deux fois)
    display(ecartes['Raison'].str.split(' ; ').explode().value_counts().to_frame('Lignes écartées'))
    # La liste des salariés écartés. max_rows=None : toute la liste est affichée, et non seulement le début et la fin
    with pd.option_context('display.max_rows', None):
        display(ecartes.sort_values('Raison'))

---
## 4. Les corrélations

C'est ici qu'on choisit les colonnes du modèle, en trois temps :

1. **la liste** des colonnes candidates : `COLONNES`, dans `reglages.py` ;
2. **la matrice et le tableau** : le lien de chaque colonne avec ce qu'on veut prédire, et les colonnes qui se répètent ;
3. **ce que chaque colonne apporte** au modèle, quand on la retire.

Quand la liste convient, on passe au notebook `2_Prediction`.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Les deux cibles et les colonnes de la liste COLONNES, en chiffres
donnees = choisir_colonnes(df)

# 2. La matrice des colonnes chiffrées. Les deux cibles sont sur les deux premières lignes
chiffrees = donnees.select_dtypes('number')
corr = chiffrees.corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, height=700,
                title='Corrélations des colonnes chiffrées')
fig.show()


def liens_et_vif(tableau_chiffre):
    # Pour chaque colonne chiffrée : son lien avec chacune des deux cibles, et son VIF.
    # Le lien est sa corrélation avec la cible, entre −1 et +1 (les deux premières lignes de la matrice).
    # Le VIF ne mesure pas l'importance : il dit si la colonne est déjà dite par les AUTRES colonnes chiffrées.
    # 1 = pas du tout ; au-dessus de 5 = à regarder ; au-dessus de 10 = elle répète les autres.
    explicatives = tableau_chiffre.drop(columns=CIBLES).astype(float)
    avec_constante = np.column_stack([np.ones(len(explicatives)), explicatives.to_numpy()])
    with np.errstate(divide='ignore', invalid='ignore'):
        vif = [variance_inflation_factor(avec_constante, i + 1) for i in range(explicatives.shape[1])]
    resultat = pd.DataFrame({'Lien avec le salaire': explicatives.corrwith(tableau_chiffre['Fixe_Annuel_MAD']),
                             'Lien avec le Compa-Ratio': explicatives.corrwith(tableau_chiffre['Compa_Ratio']), 'VIF': vif})
    # Du lien le plus fort au plus faible avec le Compa-Ratio
    return resultat.sort_values('Lien avec le Compa-Ratio', key=abs, ascending=False)


display(liens_et_vif(chiffrees).round(2))

Le tableau aide à choisir les colonnes :

* **Lien avec le salaire** et **lien avec le Compa-Ratio** : plus il est loin de 0, plus la colonne compte pour cette cible.
* **VIF** : il dit si la colonne répète les autres colonnes explicatives. 1 = non ; au-dessus de 5 = à regarder ; au-dessus de 10 = oui.

**Comprendre le VIF.** Pour chaque colonne, on essaie de la deviner à partir de toutes les autres colonnes explicatives. Le R² de cet essai dit quelle part de la colonne on arrive à deviner, et :

$$\text{VIF} = \frac{1}{1 - R^2}$$

C'est comme une corrélation, mais « 1 contre toutes » au lieu de « 2 à 2 ».

Quand il n'y a que deux colonnes, les deux mesures disent la même chose : le VIF vaut 1 / (1 − r²), avec le r qu'on lit dans la matrice. Une corrélation de 0,9 entre deux colonnes donne un VIF de 5,3.

La différence apparaît dès qu'il y a plus de deux colonnes : le VIF peut être élevé alors qu'aucune case de la matrice ne l'est, parce que la colonne se devine avec une combinaison des autres, et pas avec une seule. Exemple : âge = âge à l'embauche + ancienneté.

On garde une colonne qui a un lien et un VIF bas. Une colonne au VIF élevé et au lien faible peut partir.

Une colonne en texte, comme le métier, n'est ni dans la matrice ni dans le tableau : les modèles ne l'utilisent pas.

Pour changer les colonnes : modifier la liste `COLONNES` dans `reglages.py`, l'enregistrer, puis relancer ce notebook depuis sa première cellule.

**Ce que chaque colonne apporte.** Le tableau du dessus regarde les colonnes une par une. Ici on regarde le modèle entier : une régression apprend le Compa-Ratio avec toutes les colonnes chiffrées de la liste, puis sans une colonne, puis sans une autre.

À chaque fois, le modèle est jugé sur des salariés qu'il n'a pas vus : on coupe les salariés en 5 paquets, on apprend sur 4, on prédit le 5e, et on recommence pour chaque paquet.

* **Ce qu'elle apporte** : ce que le R² perd quand on retire la colonne, en %. Avec un R² de 0,45 qui tombe à 0,40 sans elle, elle apporte 5 %. Plus c'est grand, plus elle compte.
* Proche de 0 : le modèle fait aussi bien sans elle. Elle peut être mise de côté par un `#` dans la liste.

In [ ]:
explicatives = chiffrees.drop(columns=CIBLES)
y_log = np.log(chiffrees['Compa_Ratio'])   # le Compa-Ratio en logarithme : quelques Compa-Ratio très hauts pèsent moins


def r2_sur_salaries_non_vus(colonnes):
    # Le R² moyen sur les 5 paquets : chaque paquet est prédit par un modèle appris sur les 4 autres
    r2 = []
    for apprend, verifie in KFold(5, shuffle=True, random_state=42).split(explicatives):
        modele = LinearRegression().fit(explicatives.iloc[apprend][colonnes], y_log.iloc[apprend])
        r2.append(r2_score(y_log.iloc[verifie], modele.predict(explicatives.iloc[verifie][colonnes])))
    return np.mean(r2)


toutes = list(explicatives.columns)
r2_toutes = r2_sur_salaries_non_vus(toutes)
print(f"R² du Compa-Ratio avec toutes les colonnes de la liste : {r2_toutes:.2f}")

# On retire chaque colonne à son tour
apport = pd.DataFrame({'R² sans elle': {c: r2_sur_salaries_non_vus([x for x in toutes if x != c]) for c in toutes}})
apport["Ce qu'elle apporte (%)"] = (r2_toutes - apport['R² sans elle']) * 100
display(apport.sort_values("Ce qu'elle apporte (%)", ascending=False).round(2))

**La décision.** Une colonne qui apporte beaucoup reste. Une colonne proche de 0 peut partir : on met un `#` devant elle dans la liste `COLONNES` de `reglages.py`, et on relance ce notebook.

Quand la liste convient, on passe au notebook `2_Prediction` : il lira la même liste.